In [ ]:
import os, shutil, random

from pathlib import Path
import numpy as np
from PIL import Image, ImageEnhance, ImageFilter, ImageOps
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers.schedules import CosineDecay
from sklearn.utils.class_weight import compute_class_weight
# ------------------------------
# GPU check
# ------------------------------
gpus = tf.config.list_physical_devices('GPU')
print(f"✅ GPU(s) available: {gpus}" if gpus else "⚠️ No GPU detected. Training will use CPU.")
# ------------------------------
# CONFIG
# ------------------------------
CONFIG = {

"work_base": "/kaggle/working/poaching-dataset",

"results_dir": "/kaggle/working/results",

"img_size": (320, 320),

"batch_size": 16,

"seed": 42,

"synthetic_count": 1200,

"train_src": {

"animal": "/kaggle/input/dataset1/animal-1/WAID-final - Copy/images/train",

"empty": "/kaggle/input/dataset1/human-1/human detection dataset/0"

},

"test_src": {

"animal": "/kaggle/input/dataset1/animal-1/WAID-final - Copy/images/test",

"empty": "/kaggle/input/dataset1/human-1/human detection dataset/0"

},

"human_cutouts": "/kaggle/input/h1cutouts/human-cutouts",

"wildlife_bg": "/kaggle/input/h1cutouts/wildlife-backgrounds"

}
np.random.seed(CONFIG["seed"])

tf.random.set_seed(CONFIG["seed"])

random.seed(CONFIG["seed"])
# ------------------------------
# CREATE WORKING DIRS
# ------------------------------
def setup_dirs(cfg):

    if os.path.exists(cfg["work_base"]):

        shutil.rmtree(cfg["work_base"])

    os.makedirs(cfg["work_base"], exist_ok=True)

    os.makedirs(cfg["results_dir"], exist_ok=True)

    for sub in ["train","test"]:

        for cls in ["animal","empty","poacher"]:

            os.makedirs(Path(cfg["work_base"])/sub/cls, exist_ok=True)

    print("✅ Working directories ready.")
setup_dirs(CONFIG)

# ------------------------------
# COPY DATA
# ------------------------------
def copy_images(src, dst):

    os.makedirs(dst, exist_ok=True)

    for p in Path(src).glob("*"):

        if p.suffix.lower() in (".jpg",".jpeg",".png"):

            shutil.copy(p,dst)
copy_images(CONFIG["train_src"]["animal"], Path(CONFIG["work_base"])/"train/animal")

copy_images(CONFIG["test_src"]["animal"], Path(CONFIG["work_base"])/"test/animal")
# ------------------------------
# SYNTHETIC IMAGE GENERATION
# ------------------------------
humans = list(Path(CONFIG["human_cutouts"]).glob("*.png"))

bgs = [p for p in Path(CONFIG["wildlife_bg"]).glob("*") if p.suffix.lower() in (".jpg",".png")]

empty_src = list(Path(CONFIG["train_src"]["empty"]).glob("*"))

assert len(humans) > 0 and len(bgs) > 0, "Check human cutouts or backgrounds!"
# Split backgrounds for poacher vs empty
poacher_bgs = random.sample(bgs, int(len(bgs)*0.6))  # 60% for poacher

empty_bgs = [b for b in bgs if b not in poacher_bgs]
def paste_synthetic(bg_path, human_path=None):

    bg = Image.open(bg_path).convert("RGBA")

    if human_path:

        hm = Image.open(human_path).convert("RGBA")

        scale = random.uniform(0.25, 0.75)

        hm = hm.resize((max(1,int(hm.width*scale)), max(1,int(hm.height*scale))), Image.LANCZOS)

        if random.random() < 0.5: hm = ImageOps.mirror(hm)

        alpha_mul = random.uniform(0.6,1.0)

        alpha = hm.split()[-1].point(lambda p: int(p*alpha_mul))

        hm.putalpha(alpha)

        x = random.randint(-int(0.15*bg.width), max(0,bg.width-hm.width+int(0.15*bg.width)))

        y = random.randint(-int(0.15*bg.height), max(0,bg.height-hm.height+int(0.15*bg.height)))

        layer = Image.new("RGBA", bg.size)

        layer.paste(hm, (x, y), hm)

        comp = Image.alpha_composite(bg, layer).convert("RGB")

    else:

        comp = bg.convert("RGB")

    if random.random() < 0.6: comp = comp.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.3,1.3)))

    comp = ImageEnhance.Brightness(comp).enhance(random.uniform(0.75,1.25))

    comp = ImageEnhance.Contrast(comp).enhance(random.uniform(0.85,1.15))

    comp = ImageEnhance.Color(comp).enhance(random.uniform(0.85,1.15))

    return comp
def generate_synthetic(count, train_prob=0.9):

    for i in range(count):

        # Poacher

        img = paste_synthetic(random.choice(poacher_bgs), random.choice(humans))

        path = f"{CONFIG['work_base']}/train/poacher/synth_{i}.jpg" if random.random()<train_prob else f"{CONFIG['work_base']}/test/poacher/synth_{i}.jpg"

        img.save(path, quality=90)

        # Empty

        src = random.choice(empty_bgs + empty_src)

        img_empty = paste_synthetic(src, None)

        path = f"{CONFIG['work_base']}/train/empty/empty_{i}.jpg" if random.random()<train_prob else f"{CONFIG['work_base']}/test/empty/empty_{i}.jpg"

        img_empty.save(path, quality=90)

    print(f"✅ Generated {count} synthetic poacher + empty images.")
generate_synthetic(CONFIG["synthetic_count"])

# ------------------------------
# DATA GENERATORS
# ------------------------------
try:

    from tensorflow.keras.applications.efficientnet_v2 import EfficientNetV2B0, preprocess_input

    EfficientNetBase = EfficientNetV2B0

except:

    from tensorflow.keras.applications.efficientnet import EfficientNetB0, preprocess_input

    EfficientNetBase = EfficientNetB0
train_aug = ImageDataGenerator(

    preprocessing_function=preprocess_input,

    rotation_range=10,

    zoom_range=0.2,

    width_shift_range=0.15,

    height_shift_range=0.15,

    brightness_range=[0.5,1.5],

    shear_range=0.1,

    horizontal_flip=True,

    fill_mode="reflect"

)

test_aug = ImageDataGenerator(preprocessing_function=preprocess_input)
train_gen = train_aug.flow_from_directory(Path(CONFIG["work_base"])/"train",

    target_size=CONFIG["img_size"],

    batch_size=CONFIG["batch_size"],

    class_mode="categorical",

    seed=CONFIG["seed"],

    shuffle=True)

test_gen = test_aug.flow_from_directory(Path(CONFIG["work_base"])/"test",

    target_size=CONFIG["img_size"],

    batch_size=CONFIG["batch_size"],

    class_mode="categorical",

    shuffle=False)
# Class weights
classes = np.unique(train_gen.classes)

cw = compute_class_weight(class_weight="balanced", classes=classes, y=train_gen.classes)

class_weight = {int(c): float(w) for c,w in zip(classes,cw)}

print("Class weights:", class_weight)
# ------------------------------
# BUILD MODEL
# ------------------------------
def build_model(cfg, train_gen):

    base = EfficientNetBase(weights="imagenet", include_top=False, input_shape=cfg["img_size"]+(3,))

    base.trainable = False

    model = models.Sequential([

        base,

        layers.GlobalAveragePooling2D(),

        layers.BatchNormalization(),

        layers.Dropout(0.35),

        layers.Dense(256, activation="relu"),

        layers.BatchNormalization(),

        layers.Dropout(0.25),

        layers.Dense(len(train_gen.class_indices), activation="softmax")

    ])

    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),

        loss="categorical_crossentropy",

        metrics=["accuracy"])

    return model, base
model, base = build_model(CONFIG, train_gen)

model.summary()
# ------------------------------
# TRAIN MODEL
# ------------------------------
BASE_EPOCHS = 5

FINE_TUNE_EPOCHS = 3
def train_model(model, base, train_gen, test_gen, cfg, class_weight):

    # Ensure results directory exists

    RESULTS_DIR = Path(cfg["results_dir"])

    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    # Stage A: frozen backbone
    chk_a = ModelCheckpoint(RESULTS_DIR / "best_a.h5", save_best_only=True, monitor="val_loss", verbose=1)
    early = EarlyStopping(patience=5, restore_best_weights=True, monitor="val_loss", verbose=1)
    history_a = model.fit(
        train_gen,
        epochs=BASE_EPOCHS,
        validation_data=test_gen,
        callbacks=[chk_a, early],
        class_weight=class_weight,
        verbose=1
    )

    # Stage B: fine-tune last 50 layers
    base.trainable = True
    for layer in base.layers[:-50]:
        layer.trainable = False
    total_steps = len(train_gen) * BASE_EPOCHS
    lr_schedule = CosineDecay(1e-5, total_steps)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    chk_b = ModelCheckpoint(RESULTS_DIR / "best_b.h5", save_best_only=True, monitor="val_loss", verbose=1)
    history_b = model.fit(
        train_gen,
        initial_epoch=BASE_EPOCHS,
        epochs=BASE_EPOCHS + FINE_TUNE_EPOCHS,
        validation_data=test_gen,
        callbacks=[chk_b, early],
        class_weight=class_weight,
        verbose=1
    )

    # Save final model
    model.save(RESULTS_DIR / "final_model.keras")
    print("✅ Model saved to:", RESULTS_DIR)

    return model, history_a, history_b
model, hist_a, hist_b = train_model(model, base, train_gen, test_gen, CONFIG, class_weight)

In [ ]:
import shutil
from IPython.display import FileLink

# Zip the final model (or any other files you want)
shutil.make_archive("/kaggle/working/final_model", 'zip', "/kaggle/working/results", "final_model.keras")

# Create a clickable download link
FileLink("/kaggle/working/final_model.zip")


In [ ]:
# ============================================================
# CELL — GET PREDICTIONS
# ============================================================

# Get predictions on test set
pred_probs = model.predict(test_gen)          # softmax probabilities
pred_classes = np.argmax(pred_probs, axis=1)  # predicted class indices
true_classes = test_gen.classes               # ground-truth labels
labels = list(test_gen.class_indices.keys())  # class names


In [ ]:
import pandas as pd
from pathlib import Path

results_dir = Path(CONFIG["results_dir"])
results_dir.mkdir(parents=True, exist_ok=True)

df_preds = pd.DataFrame({
    "filename": [Path(f).name for f in test_gen.filepaths],
    "true": [labels[i] for i in true_classes],
    "pred": [labels[i] for i in pred_classes],
    "prob": pred_probs.max(axis=1)
})

df_preds.to_csv(results_dir/"test_predictions.csv", index=False)
print("✅ Predictions saved to CSV.")


In [ ]:
# ============================================================
# CELL — FULL EVALUATION DASHBOARD
# ============================================================

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, precision_recall_curve, average_precision_score
from sklearn.preprocessing import label_binarize
from PIL import Image
from IPython.display import display
import pandas as pd
from pathlib import Path

# ------------------------------
# Confusion Matrix (Normalized + Counts)
# ------------------------------
cm = confusion_matrix(true_classes, pred_classes)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(8,6))
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", cbar=True, xticklabels=labels, yticklabels=labels)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j+0.5, i+0.5, f"{cm[i,j]}", color='red', ha='center', va='center')
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title("Normalized Confusion Matrix with Counts")
plt.show()

# ------------------------------
# Per-Class ROC Curves
# ------------------------------
y_true_bin = label_binarize(true_classes, classes=range(len(labels)))
plt.figure(figsize=(7,6))
for i, label in enumerate(labels):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], pred_probs[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{label} (AUC={roc_auc:.2f})")
plt.plot([0,1], [0,1], 'k--', lw=1)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves per Class")
plt.legend()
plt.show()

# ------------------------------
# Precision-Recall Curves
# ------------------------------
plt.figure(figsize=(7,6))
for i, label in enumerate(labels):
    precision, recall, _ = precision_recall_curve(y_true_bin[:, i], pred_probs[:, i])
    ap = average_precision_score(y_true_bin[:, i], pred_probs[:, i])
    plt.plot(recall, precision, label=f"{label} (AP={ap:.2f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curves per Class")
plt.legend()
plt.show()

# ------------------------------
# Metrics Heatmap
# ------------------------------
report = classification_report(true_classes, pred_classes, target_names=labels, output_dict=True)
df_metrics = pd.DataFrame(report).transpose()[['precision','recall','f1-score','support']]
plt.figure(figsize=(6,4))
sns.heatmap(df_metrics[['precision','recall','f1-score']], annot=True, fmt=".2f", cmap="YlGnBu", cbar=True)
plt.title("Per-Class Metrics Heatmap")
plt.show()
print(df_metrics)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
import random

# Sample 20 random predictions
sample_imgs = df_preds.sample(20).reset_index(drop=True)

# Grid size
rows, cols = 4, 5
plt.figure(figsize=(4*cols, 4*rows))

for i, row in sample_imgs.iterrows():
    img_path = Path(CONFIG["work_base"]) / "test" / row['true'] / row['filename']
    if img_path.exists():
        img = Image.open(img_path)
        plt.subplot(rows, cols, i+1)
        plt.imshow(img)
        plt.axis('off')
        plt.title(f"True: {row['true']}\nPred: {row['pred']}\nProb: {row['prob']:.2f}", fontsize=9)
    else:
        print(f"❌ Missing: {img_path}")

plt.suptitle("Sample 20 Test Images", fontsize=16)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.cm as cm

from tensorflow.keras.preprocessing.image import load_img, img_to_array
def safe_grad_cam_grid(base_model, img_paths, grid_size=(4,5), alpha=0.4):

    rows, cols = grid_size

    plt.figure(figsize=(4*cols, 4*rows))
    # Find last Conv2D layer
    conv_layers = [l for l in base_model.layers if isinstance(l, tf.keras.layers.Conv2D)]
    if not conv_layers:
        raise ValueError("❌ No Conv2D layers found in base model!")
    last_conv = conv_layers[-1]
    print(f"📌 Last Conv layer used for Grad-CAM: {last_conv.name}")

    grad_model = tf.keras.Model(inputs=base_model.input, outputs=last_conv.output)

    for i, fp in enumerate(img_paths):
        if i >= rows * cols: break

        img = load_img(fp)
        arr = img_to_array(img)
        arr = tf.image.resize(arr, CONFIG["img_size"]).numpy()
        x = np.expand_dims(arr, axis=0).astype("float32")
        x = preprocess_input(x)

        with tf.GradientTape() as tape:
            conv_output = grad_model(x)
            # The loss is the maximum activation of the output feature map,
            # which targets the most relevant feature regardless of class.
            loss = tf.reduce_max(conv_output)
        
        # Compute gradients of the loss with respect to the output feature map
        grads = tape.gradient(loss, conv_output)
        
        # Global average pool the gradients to get a weight for each feature map channel
        pooled_grads = tf.reduce_mean(grads, axis=(0,1,2))
        
        # Get the feature map output for the current image
        conv_output = conv_output[0]

        # Compute the heatmap: weighted sum of feature maps by their gradient weights
        heatmap = tf.reduce_sum(pooled_grads * conv_output, axis=-1)
        
        # Apply ReLU to keep only positive contributions
        heatmap = np.maximum(heatmap, 0)
        
        # Normalize the heatmap
        heatmap /= (np.max(heatmap) + 1e-8)

        # Convert to 8-bit integer and apply the 'jet' colormap
        heatmap_uint8 = np.uint8(255*heatmap)
        jet_colors = cm.get_cmap("jet")(np.arange(256))[:, :3]
        jet_heatmap = jet_colors[heatmap_uint8]
        
        # Resize the heatmap to the original image size
        jet_heatmap = tf.image.resize(jet_heatmap, CONFIG["img_size"]).numpy()

        # Superimpose the heatmap onto the original image
        superimposed = np.uint8(jet_heatmap*255*alpha + arr*(1-alpha))

        # Plotting
        plt.subplot(rows, cols, i+1)
        plt.imshow(superimposed.astype("uint8"))
        plt.axis("off")
        plt.title(Path(fp).parent.name, fontsize=9)

    plt.suptitle("Grad-CAM Visualization (Base Model Only)", fontsize=16)
    plt.tight_layout()
    plt.show()

# Pick 20 random test images
sample_files_20 = np.random.choice(test_gen.filepaths, size=min(20, len(test_gen.filepaths)), replace=False)

# Run Grad-CAM grid
safe_grad_cam_grid(base, sample_files_20, grid_size=(4,5), alpha=0.4)